# EEG_08 — Clustering dei Soggetti da Caratteristiche del Segnale

**Approccio data-first (Francesco)**:
1. Estrai feature dal segnale EEG grezzo per soggetto (potenza per banda, connettività PCC, metriche grafo)
2. Clustering dei soggetti nello spazio feature (K-means + gerarchico)
3. **Solo dopo**: confronto con le performance dei modelli (EEG_06 results)

Obiettivo: capire se i soggetti hanno caratteristiche EEG sistematicamente diverse
e se questi gruppi si riflettono nelle performance dei modelli.

---
**Reference**: Iacomi et al. 2026 — gamma dominanza, 3 pathways stabili (stesso dataset)  
**Env**: `daniele_311` (Python 3.11)

In [ ]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

# Bande di frequenza da analizzare (Hz)
FREQ_BANDS = {
    "delta": (1,   4),
    "theta": (4,   8),
    "alpha": (8,  13),
    "beta":  (13, 30),
    "gamma": (30, 60),   # Iacomi 2026: banda più informativa per IS
}

FS = 256          # frequenza di campionamento (Hz)
N_SAMPLES = 384   # campioni per trial (~1.5s)
N_CHANS   = 61    # canali EEG nel file H5 / CSV

# Numero massimo di trial per soggetto per l'analisi (None = tutti)
MAX_TRIALS_PER_SUBJ = None

# K per grafo PCC k-NN (usato per metriche di grafo)
K_GRAPH = 6

# Numero di componenti PCA da mantenere prima di UMAP / clustering
N_PCA_COMPONENTS = 20

# Clustering: range di k per K-means
K_RANGE = [2, 3, 4, 5]

# Seed riproducibilità
RANDOM_STATE = 42

# Percorso CSV risultati EEG_06 (facoltativo — se esiste, mostra overlap con accuracy)
EEG06_RESULTS_CSV = "../data/interim/eeg06_results.csv"   # modifica se diverso

print("Config OK")

In [ ]:
# ============================================================
# IMPORT E PATHS
# ============================================================

import os, sys, json, warnings
warnings.filterwarnings("ignore")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict

from scipy import signal as scipy_signal
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import f_oneway, kruskal

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("⚠️  umap-learn non installato — skip UMAP, solo PCA.")
    print("   Installa con: pip install umap-learn")

# ---- Project root ----
project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)
sys.path.insert(0, str(project_root / "scripts"))

# ---- Paths ----
CSV_ROOT    = project_root / "data" / "raw_csv" / "training_set"
INTERIM_DIR = project_root / "data" / "interim"
FIGURES_DIR = project_root / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ---- Canali ----
_eloc_candidates = [
    INTERIM_DIR / "ebneuro.csv",
    Path("/mnt/c/Users/students/Desktop/Paolo/LM_Thesis/ebneuro.csv"),
]
CH_NAMES = [str(i) for i in range(N_CHANS)]
for _p in _eloc_candidates:
    if _p.exists():
        CH_NAMES = pd.read_csv(_p, sep=";", decimal=",")["labels"].tolist()[:N_CHANS]
        print(f"Canali caricati da: {_p.name}")
        break

print(f"Project root : {project_root}")
print(f"CSV root     : {CSV_ROOT}")
print(f"N canali     : {N_CHANS}")

In [ ]:
# ============================================================
# CARICAMENTO TRIAL E ESTRAZIONE FEATURE PER SOGGETTO
# ============================================================
# Feature estratte per ogni soggetto (media su tutti i trial del soggetto):
#   - Potenza relativa per banda (δ, θ, α, β, γ) × N_CHANS
#   - Matrice PCC media (N_CHANS × N_CHANS) → vettore triangolo superiore
#   - Metriche grafo: grado medio, clustering coefficient medio, densità
# ============================================================

def load_csv_trial(csv_path: Path) -> np.ndarray:
    """Legge CSV 61×384 senza header. Restituisce float32 (61, 384)."""
    return pd.read_csv(csv_path, header=None).values.astype(np.float32)


def band_power_per_channel(x_np: np.ndarray, fs: int, bands: dict) -> np.ndarray:
    """
    Potenza relativa per banda via Welch per ogni canale.

    Args:
        x_np  : (N_CHANS, N_SAMPLES)
        fs    : frequenza campionamento
        bands : dict {nome: (f_low, f_high)}

    Returns:
        feat: (N_CHANS, N_BANDS) — potenza relativa
    """
    freqs, psd = scipy_signal.welch(x_np, fs=fs, nperseg=min(128, x_np.shape[1]))
    # psd shape: (N_CHANS, n_freqs)
    total_power = psd.sum(axis=1, keepdims=True) + 1e-12
    feat = []
    for (f_lo, f_hi) in bands.values():
        mask = (freqs >= f_lo) & (freqs < f_hi)
        band_p = psd[:, mask].sum(axis=1) / total_power[:, 0]
        feat.append(band_p)
    return np.stack(feat, axis=1)  # (N_CHANS, N_BANDS)


def pcc_matrix(x_np: np.ndarray) -> np.ndarray:
    """Pearson |PCC| tra canali. Shape: (N, N)."""
    pcc = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0.0)
    return pcc


def graph_metrics_from_pcc(pcc: np.ndarray, k: int) -> dict:
    """
    Metriche scalari da grafo k-NN PCC:
      - mean_degree       : grado medio
      - density           : densità del grafo
      - mean_clustering   : coefficiente clustering medio
      - mean_strength     : forza media (somma pesi vicini)
    """
    N = pcc.shape[0]
    # k-NN adjacency
    adj = np.zeros((N, N))
    for i in range(N):
        row = pcc[i].copy(); row[i] = -1.0
        top_k = np.argsort(row)[-k:]
        adj[i, top_k] = pcc[i, top_k]
        adj[top_k, i] = pcc[top_k, i]

    # Grado (binarizzato)
    adj_bin  = (adj > 0).astype(float)
    degrees  = adj_bin.sum(axis=1)
    # Densità
    density = adj_bin.sum() / (N * (N - 1))
    # Coefficiente clustering (formula pesata semplificata)
    cc = []
    for i in range(N):
        nbrs = np.where(adj_bin[i] > 0)[0]
        ki = len(nbrs)
        if ki < 2:
            cc.append(0.0)
            continue
        triangles = sum(
            adj_bin[u, v]
            for idx_u, u in enumerate(nbrs)
            for v in nbrs[idx_u+1:]
        )
        cc.append(2 * triangles / (ki * (ki - 1)))
    # Forza media (pesi)
    strength = adj.sum(axis=1)

    return {
        "mean_degree":     float(degrees.mean()),
        "density":         float(density),
        "mean_clustering": float(np.mean(cc)),
        "mean_strength":   float(strength.mean()),
    }


# ---- Scansione cartelle ----
session_dirs = sorted(CSV_ROOT.iterdir())
subjects_all = sorted(set(d.name.split("_")[0] for d in session_dirs if d.is_dir()))
print(f"Soggetti trovati: {len(subjects_all)}")
print(f"Sessioni totali : {len(session_dirs)}")
print(f"Feature per soggetto:")
n_bands  = len(FREQ_BANDS)
n_triu   = N_CHANS * (N_CHANS - 1) // 2
n_graph  = 4  # mean_degree, density, mean_clustering, mean_strength
n_feat_total = N_CHANS * n_bands + n_triu + n_graph
print(f"  Potenza per banda : {N_CHANS} canali × {n_bands} bande = {N_CHANS*n_bands}")
print(f"  PCC triangolo sup : {n_triu}")
print(f"  Metriche grafo    : {n_graph}")
print(f"  TOTALE            : {n_feat_total}")

In [ ]:
# ============================================================
# ESTRAZIONE FEATURE: itera tutti i soggetti
# Salva/carica cache in data/interim/subject_features.npz
# ============================================================

CACHE_PATH = INTERIM_DIR / "subject_features.npz"
FORCE_RECOMPUTE = False   # True = ricalcola anche se cache esiste

if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    print(f"Carico da cache: {CACHE_PATH}")
    data = np.load(CACHE_PATH, allow_pickle=True)
    subject_ids  = list(data["subject_ids"])
    X_subjects   = data["X_subjects"]       # (N_subj, n_feat)
    feat_names   = list(data["feat_names"])
    n_trials_per = dict(zip(data["subject_ids"], data["n_trials_per"]))
    # Band power matrix: (N_subj, N_CHANS, N_BANDS)
    band_power_mat = data["band_power_mat"]
    # PCC medio per soggetto: (N_subj, N_CHANS, N_CHANS)
    pcc_mean_mat = data["pcc_mean_mat"]
    print(f"  Soggetti : {len(subject_ids)}")
    print(f"  Feature  : {X_subjects.shape[1]}")
else:
    print("Estrazione feature per soggetto...")

    subject_ids     = []
    X_list          = []         # lista righe feature
    band_power_list = []         # (N_CHANS, N_BANDS) medi
    pcc_mean_list   = []         # (N_CHANS, N_CHANS) medi
    n_trials_per    = {}

    for subj in tqdm(subjects_all, desc="Soggetti"):
        # Raccogli tutti i trial del soggetto
        bp_accum  = []    # band power accumulatore
        pcc_accum = []    # PCC accumulatore
        g_metrics = defaultdict(list)

        subj_dirs = sorted(d for d in session_dirs if d.name.startswith(subj + "_"))
        trial_count = 0

        for sess_dir in subj_dirs:
            csv_files = sorted(sess_dir.glob("*.csv"))
            for csv_path in csv_files:
                if MAX_TRIALS_PER_SUBJ and trial_count >= MAX_TRIALS_PER_SUBJ:
                    break
                x = load_csv_trial(csv_path)       # (61, 384)
                if x.shape != (N_CHANS, N_SAMPLES):
                    continue
                # Band power
                bp = band_power_per_channel(x, FS, FREQ_BANDS)  # (61, 5)
                bp_accum.append(bp)
                # PCC
                pcc = pcc_matrix(x)    # (61, 61)
                pcc_accum.append(pcc)
                # Graph metrics
                gm = graph_metrics_from_pcc(pcc, k=K_GRAPH)
                for k_m, v_m in gm.items():
                    g_metrics[k_m].append(v_m)
                trial_count += 1

        if trial_count == 0:
            continue

        n_trials_per[subj] = trial_count

        # Medie sul soggetto
        bp_mean  = np.mean(bp_accum,  axis=0)   # (61, 5)
        pcc_mean = np.mean(pcc_accum, axis=0)   # (61, 61)

        # ---- Costruisci vettore feature ----
        # 1. Band power flattened: (61*5,) = 305
        feat_bp   = bp_mean.flatten()
        # 2. PCC triangolo superiore: N_CHANS*(N_CHANS-1)/2 = 1830
        feat_pcc  = pcc_mean[np.triu_indices(N_CHANS, k=1)]
        # 3. Graph metrics scalari: 4
        feat_gm   = np.array([np.mean(g_metrics[k_m]) for k_m in
                               ["mean_degree", "density", "mean_clustering", "mean_strength"]])

        X_list.append(np.concatenate([feat_bp, feat_pcc, feat_gm]))
        subject_ids.append(subj)
        band_power_list.append(bp_mean)
        pcc_mean_list.append(pcc_mean)

    X_subjects     = np.array(X_list)          # (N_subj, n_feat)
    band_power_mat = np.array(band_power_list) # (N_subj, N_CHANS, N_BANDS)
    pcc_mean_mat   = np.array(pcc_mean_list)   # (N_subj, N_CHANS, N_CHANS)

    # Nomi feature
    band_names = list(FREQ_BANDS.keys())
    feat_names = (
        [f"bp_{b}_{ch}" for b in band_names for ch in range(N_CHANS)] +
        [f"pcc_{i}_{j}" for i, j in zip(*np.triu_indices(N_CHANS, k=1))] +
        ["mean_degree", "density", "mean_clustering", "mean_strength"]
    )

    # Salva cache
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(
        CACHE_PATH,
        subject_ids   = np.array(subject_ids),
        X_subjects    = X_subjects,
        feat_names    = np.array(feat_names),
        n_trials_per  = np.array([n_trials_per[s] for s in subject_ids]),
        band_power_mat= band_power_mat,
        pcc_mean_mat  = pcc_mean_mat,
    )
    print(f"\nSalvato in cache: {CACHE_PATH}")

print(f"\nSoggetti processati : {len(subject_ids)}")
print(f"Feature per soggetto: {X_subjects.shape[1]}")
print(f"Trial medi per sogg : {np.mean(list(n_trials_per.values())):.0f}")
print(f"Shape X_subjects    : {X_subjects.shape}")

In [ ]:
# ============================================================
# ANALISI POTENZA PER BANDA — panoramica tra soggetti
# Ispirati da Iacomi 2026: gamma è la banda più integrata per IS
# ============================================================

band_names = list(FREQ_BANDS.keys())
n_subj     = len(subject_ids)

# ---- 1. Potenza media per banda (media su canali e trial) ----
bp_mean_per_subj_band = band_power_mat.mean(axis=1)  # (N_subj, N_BANDS)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Potenza Relativa per Banda EEG — tutti i soggetti", fontsize=13)

# Heatmap soggetti × bande
im0 = axes[0].imshow(bp_mean_per_subj_band, aspect="auto", cmap="viridis")
axes[0].set_xticks(range(len(band_names))); axes[0].set_xticklabels(band_names)
axes[0].set_yticks(range(0, n_subj, max(1, n_subj // 10)))
axes[0].set_yticklabels([subject_ids[i] for i in range(0, n_subj, max(1, n_subj // 10))])
axes[0].set_xlabel("Banda"); axes[0].set_ylabel("Soggetto")
axes[0].set_title("Heatmap: potenza per banda × soggetto")
plt.colorbar(im0, ax=axes[0], label="Potenza relativa")

# Boxplot per banda (distribuzione tra soggetti)
axes[1].boxplot(bp_mean_per_subj_band, labels=band_names, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.7))
axes[1].set_xlabel("Banda"); axes[1].set_ylabel("Potenza relativa")
axes[1].set_title("Distribuzione inter-soggetto per banda")
axes[1].grid(axis="y", alpha=0.4)

# Coefficiente di variazione (CV) per banda: quanto variano i soggetti?
cv_per_band = bp_mean_per_subj_band.std(axis=0) / (bp_mean_per_subj_band.mean(axis=0) + 1e-9)
axes[2].bar(band_names, cv_per_band, color=["royalblue", "steelblue", "deepskyblue", "darkorange", "crimson"])
axes[2].set_xlabel("Banda"); axes[2].set_ylabel("CV (std/mean)")
axes[2].set_title("Variabilità inter-soggetto per banda\n(alto CV = maggior discriminazione)")
axes[2].grid(axis="y", alpha=0.4)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_band_power_overview.png", dpi=120, bbox_inches="tight")
plt.show()

# Test ANOVA: varianza inter-soggetto per banda
print("\nANOVA / Kruskal-Wallis: varianza inter-banda (potenza media per soggetto)")
for b_idx, b_name in enumerate(band_names):
    vals = bp_mean_per_subj_band[:, b_idx]
    print(f"  {b_name:6s}:  μ={vals.mean():.4f}  σ={vals.std():.4f}  CV={cv_per_band[b_idx]:.3f}")

_, p_kw = kruskal(*[bp_mean_per_subj_band[:, i] for i in range(len(band_names))])
print(f"\nKruskal-Wallis (differenza tra bande): p={p_kw:.4f}")

In [ ]:
# ============================================================
# ANALISI PCC MEDIO — connettività media per soggetto
# ============================================================

# PCC medio per soggetto già in pcc_mean_mat: (N_subj, N_CHANS, N_CHANS)
# Indice scalare: connettività totale media del soggetto
mean_connectivity = np.array([
    pcc_mean_mat[i][np.triu_indices(N_CHANS, k=1)].mean()
    for i in range(n_subj)
])
# Deviazione standard della connettività (variabilità intra-soggetto)
std_connectivity = np.array([
    pcc_mean_mat[i][np.triu_indices(N_CHANS, k=1)].std()
    for i in range(n_subj)
])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Connettività PCC Media per Soggetto", fontsize=13)

# Distribuzione connettività media
axes[0].hist(mean_connectivity, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(mean_connectivity.mean(), color="red", ls="--", lw=2,
                label=f"μ={mean_connectivity.mean():.3f}")
axes[0].set_xlabel("Connettività PCC media"); axes[0].set_ylabel("N soggetti")
axes[0].set_title("Distribuzione connettività media")
axes[0].legend()

# Scatter: μ vs σ connettività
axes[1].scatter(mean_connectivity, std_connectivity,
                c=mean_connectivity, cmap="plasma", s=80, alpha=0.8, edgecolors="black", lw=0.5)
for i, sid in enumerate(subject_ids):
    axes[1].annotate(sid.replace("P", ""), (mean_connectivity[i], std_connectivity[i]),
                     fontsize=6, ha="center", va="bottom")
axes[1].set_xlabel("Connettività media (μ PCC)")
axes[1].set_ylabel("Variabilità connettività (σ PCC)")
axes[1].set_title("Profilo connettività per soggetto")

# PCC medio su tutti i soggetti
pcc_global = pcc_mean_mat.mean(axis=0)  # (N_CHANS, N_CHANS)
im2 = axes[2].imshow(pcc_global, cmap="RdYlBu_r", vmin=0, vmax=0.5)
axes[2].set_title("PCC globale (media 70 soggetti)")
axes[2].set_xlabel("Canale"); axes[2].set_ylabel("Canale")
plt.colorbar(im2, ax=axes[2], label="|PCC|")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_pcc_connectivity.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Connettività media globale: μ={mean_connectivity.mean():.4f}  σ={mean_connectivity.std():.4f}")
print(f"Soggetti ad alta connettività (>μ+σ): {(mean_connectivity > mean_connectivity.mean()+mean_connectivity.std()).sum()}")
print(f"Soggetti a bassa connettività (<μ-σ): {(mean_connectivity < mean_connectivity.mean()-mean_connectivity.std()).sum()}")

In [ ]:
# ============================================================
# NORMALIZZAZIONE + PCA
# ============================================================

# Standardizzazione Z-score per feature
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_subjects)  # (N_subj, n_feat)

# Gestione NaN / Inf (possibili se un soggetto ha pochi trial)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# PCA
n_pca = min(N_PCA_COMPONENTS, X_scaled.shape[0] - 1, X_scaled.shape[1])
pca = PCA(n_components=n_pca, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)  # (N_subj, n_pca)

explained = pca.explained_variance_ratio_
cum_var   = explained.cumsum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("PCA: varianza spiegata", fontsize=12)

axes[0].bar(range(1, n_pca + 1), explained * 100, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Componente"); axes[0].set_ylabel("Varianza spiegata (%)")
axes[0].set_title("Varianza per componente")
axes[0].set_xticks(range(1, n_pca + 1, max(1, n_pca // 10)))

axes[1].plot(range(1, n_pca + 1), cum_var * 100, "o-", color="crimson", markersize=4)
axes[1].axhline(80, color="gray", ls="--", label="80%")
axes[1].axhline(95, color="black", ls="--", label="95%")
axes[1].set_xlabel("N componenti"); axes[1].set_ylabel("Varianza cumulativa (%)")
axes[1].set_title("Varianza cumulativa"); axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_pca_variance.png", dpi=120, bbox_inches="tight")
plt.show()

n_for_80  = int(np.searchsorted(cum_var, 0.80)) + 1
n_for_95  = int(np.searchsorted(cum_var, 0.95)) + 1
print(f"PC1+PC2 spiegano    : {cum_var[1]*100:.1f}% della varianza")
print(f"PC per 80% varianza : {n_for_80}")
print(f"PC per 95% varianza : {n_for_95}")

In [ ]:
# ============================================================
# UMAP + PCA2D: visualizzazione nello spazio 2D
# ============================================================

# PCA a 2 componenti per visualizzazione
pca2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca2d = pca2d.fit_transform(X_scaled)  # (N_subj, 2)

if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, n_neighbors=min(15, n_subj - 1),
                        random_state=RANDOM_STATE, min_dist=0.1)
    X_umap = reducer.fit_transform(X_pca)  # applica UMAP su PCA
else:
    X_umap = None

# ---- Visualizzazione colorata per connettività media ----
n_plots = 2 if X_umap is not None else 1
fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 6))
if n_plots == 1:
    axes = [axes]
fig.suptitle("Spazio soggetti: PCA e UMAP", fontsize=13)

for ax, (proj, title) in zip(axes, [(X_pca2d, "PCA 2D"), (X_umap, "UMAP 2D")]):
    if proj is None:
        continue
    sc = ax.scatter(proj[:, 0], proj[:, 1],
                    c=mean_connectivity, cmap="plasma", s=80,
                    edgecolors="black", linewidths=0.5, alpha=0.9)
    plt.colorbar(sc, ax=ax, label="Connettività PCC media")
    for i, sid in enumerate(subject_ids):
        ax.annotate(sid.replace("P", ""), (proj[i, 0], proj[i, 1]),
                    fontsize=6, ha="center", va="bottom")
    ax.set_xlabel(f"{title} dim 1"); ax.set_ylabel(f"{title} dim 2")
    ax.set_title(f"{title} — colorato per connettività PCC")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_embedding_2d.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CLUSTERING K-MEANS: k = 2, 3, 4, 5
# ============================================================

# Input clustering: X_pca (n_PCA componenti)
X_clust = X_pca

kmeans_results = {}
silhouette_scores = {}

print("K-Means clustering:")
print(f"{'k':>3}  {'Silhouette':>12}  {'Inertia':>12}")
print("-" * 32)

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_clust)
    if len(set(labels)) > 1:
        sil = silhouette_score(X_clust, labels)
    else:
        sil = 0.0
    kmeans_results[k]     = labels
    silhouette_scores[k]  = sil
    sizes = [int((labels == c).sum()) for c in range(k)]
    print(f"{k:>3}  {sil:>12.4f}  {km.inertia_:>12.1f}   cluster_sizes={sizes}")

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k (silhouette): {best_k}  (score={silhouette_scores[best_k]:.4f})")

# ---- Plot silhouette per k ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_RANGE, [silhouette_scores[k] for k in K_RANGE], "o-", color="steelblue", markersize=8)
axes[0].axvline(best_k, color="red", ls="--", label=f"Best k={best_k}")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Silhouette score")
axes[0].set_title("Silhouette score per k"); axes[0].legend()
axes[0].grid(alpha=0.4)

# Silhouette samples per best_k
best_labels = kmeans_results[best_k]
sil_samples = silhouette_samples(X_clust, best_labels)
y_lower = 10
colors   = cm.tab10(np.arange(best_k) / best_k)
for c_id in range(best_k):
    c_sil = np.sort(sil_samples[best_labels == c_id])
    y_upper = y_lower + len(c_sil)
    axes[1].fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                           facecolor=colors[c_id], edgecolor="none", alpha=0.8,
                           label=f"Cluster {c_id} (n={len(c_sil)})")
    y_lower = y_upper + 5
axes[1].axvline(silhouette_scores[best_k], color="red", ls="--", lw=1.5)
axes[1].set_xlabel("Silhouette coefficient"); axes[1].set_ylabel("Soggetti per cluster")
axes[1].set_title(f"Silhouette plot — k={best_k}"); axes[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_kmeans_silhouette.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CLUSTERING GERARCHICO + DENDROGRAMMA
# ============================================================

# Linkage su X_pca (Ward)
Z = linkage(X_clust, method="ward", metric="euclidean")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Clustering Gerarchico dei Soggetti", fontsize=13)

# Dendrogramma
dend = dendrogram(
    Z, labels=subject_ids, ax=axes[0],
    leaf_rotation=90, leaf_font_size=6,
    color_threshold=0.7 * max(Z[:, 2]),
)
axes[0].set_title(f"Dendrogramma (Ward linkage, n={n_subj})")
axes[0].set_ylabel("Distanza")

# Confronto flat clustering gerarchico con K-Means al best_k
hier_labels = fcluster(Z, t=best_k, criterion="maxclust") - 1  # 0-indexed
km_labels   = kmeans_results[best_k]

# Heatmap accordo K-Means vs gerarchico
from sklearn.metrics import confusion_matrix
conf = confusion_matrix(km_labels, hier_labels)
sns.heatmap(conf, annot=True, fmt="d", ax=axes[1], cmap="Blues",
            xticklabels=[f"Hier_{c}" for c in range(best_k)],
            yticklabels=[f"KM_{c}"   for c in range(best_k)])
axes[1].set_title(f"K-Means vs Gerarchico (k={best_k})")
axes[1].set_xlabel("Cluster gerarchico")
axes[1].set_ylabel("Cluster K-Means")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_hierarchical_clustering.png", dpi=120, bbox_inches="tight")
plt.show()

# Accordo (Adjusted Rand Index)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(km_labels, hier_labels)
print(f"Adjusted Rand Index (K-Means vs Gerarchico, k={best_k}): {ari:.3f}")
print("  (1.0 = accordo perfetto, 0.0 = casuale)")

In [ ]:
# ============================================================
# VISUALIZZAZIONE CLUSTER NELLO SPAZIO 2D (PCA + UMAP)
# ============================================================

# Usa best_k come cluster finale
FINAL_CLUSTER_LABELS = kmeans_results[best_k]   # array (N_subj,)
CLUSTER_COLORS = plt.cm.tab10(np.arange(best_k) / best_k)

# Aggiungi colore connettività media come dimensione extra
fig_cols = 2 + (1 if X_umap is not None else 0)
fig, axes = plt.subplots(1, fig_cols, figsize=(7 * fig_cols, 6))
fig.suptitle(f"Cluster Soggetti (K-Means k={best_k})", fontsize=13)

projs = [(X_pca2d, "PCA 2D")]
if X_umap is not None:
    projs.append((X_umap, "UMAP 2D"))

for ax_idx, (proj, title) in enumerate(projs):
    ax = axes[ax_idx]
    for c_id in range(best_k):
        mask = FINAL_CLUSTER_LABELS == c_id
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   s=100, label=f"Cluster {c_id} (n={mask.sum()})",
                   color=CLUSTER_COLORS[c_id], edgecolors="black", linewidths=0.5, alpha=0.85)
    for i, sid in enumerate(subject_ids):
        ax.annotate(sid.replace("P", ""), (proj[i, 0], proj[i, 1]),
                    fontsize=6, ha="center", va="bottom")
    ax.set_xlabel(f"{title} dim 1"); ax.set_ylabel(f"{title} dim 2")
    ax.set_title(f"{title} — cluster soggetti")
    ax.legend(fontsize=8)

# Plot: feature discriminanti per cluster (potenza gamma)
ax_feat = axes[-1]
gamma_idx = list(FREQ_BANDS.keys()).index("gamma")
bp_gamma_per_subj = band_power_mat[:, :, gamma_idx].mean(axis=1)  # (N_subj,)

for c_id in range(best_k):
    mask = FINAL_CLUSTER_LABELS == c_id
    ax_feat.scatter(mean_connectivity[mask], bp_gamma_per_subj[mask],
                    s=80, label=f"Cluster {c_id}",
                    color=CLUSTER_COLORS[c_id], edgecolors="black", lw=0.5, alpha=0.85)
ax_feat.set_xlabel("Connettività PCC media")
ax_feat.set_ylabel("Potenza gamma media")
ax_feat.set_title("Connettività vs Gamma\n(Iacomi: gamma più integrata per IS)")
ax_feat.legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_cluster_visualization.png", dpi=120, bbox_inches="tight")
plt.show()

# Riepilogo cluster
print(f"\nRiepilogo cluster (k={best_k}):")
print(f"{'Cluster':>8}  {'N':>4}  {'Conn.μ':>8}  {'Conn.σ':>8}  {'Gamma':>8}  {'Beta':>8}")
beta_idx = list(FREQ_BANDS.keys()).index("beta")
bp_beta_per_subj = band_power_mat[:, :, beta_idx].mean(axis=1)
for c_id in range(best_k):
    mask = FINAL_CLUSTER_LABELS == c_id
    print(f"{c_id:>8}  {mask.sum():>4}  "
          f"{mean_connectivity[mask].mean():>8.4f}  "
          f"{mean_connectivity[mask].std():>8.4f}  "
          f"{bp_gamma_per_subj[mask].mean():>8.4f}  "
          f"{bp_beta_per_subj[mask].mean():>8.4f}")

In [ ]:
# ============================================================
# PROFILO BANDE PER CLUSTER
# Ispirato da Iacomi 2026: analisi per banda separata
# ============================================================

band_names = list(FREQ_BANDS.keys())
n_bands    = len(band_names)

# Potenza media per banda, per cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Profilo Bande per Cluster (k={best_k})", fontsize=12)

# Grouped boxplot per banda e cluster
data_box = []
for b_idx, b_name in enumerate(band_names):
    for c_id in range(best_k):
        mask = FINAL_CLUSTER_LABELS == c_id
        for val in band_power_mat[mask, :, b_idx].mean(axis=1):
            data_box.append({"Banda": b_name, "Cluster": f"C{c_id}", "Potenza": val})
df_box = pd.DataFrame(data_box)

cluster_palette = {f"C{c_id}": CLUSTER_COLORS[c_id] for c_id in range(best_k)}
sns.boxplot(data=df_box, x="Banda", y="Potenza", hue="Cluster",
            palette=cluster_palette, ax=axes[0])
axes[0].set_title("Potenza relativa per banda × cluster")
axes[0].set_xlabel("Banda"); axes[0].set_ylabel("Potenza relativa")
axes[0].legend(title="Cluster", fontsize=8)
axes[0].grid(axis="y", alpha=0.4)

# Radar chart: profilo medio per cluster
from matplotlib.patches import FancyArrowPatch
angles   = np.linspace(0, 2 * np.pi, n_bands, endpoint=False)
angles   = np.concatenate([angles, [angles[0]]])  # chiudi il cerchio

ax_radar = plt.subplot(122, projection="polar") if False else axes[1]
# Fallback: se non polar, usa lineare
cluster_bp_profiles = []
for c_id in range(best_k):
    mask = FINAL_CLUSTER_LABELS == c_id
    profile = band_power_mat[mask, :, :].mean(axis=(0, 1))  # (N_BANDS,)
    cluster_bp_profiles.append(profile)

x_pos = np.arange(n_bands)
width = 0.8 / best_k
for c_id, profile in enumerate(cluster_bp_profiles):
    axes[1].bar(x_pos + c_id * width, profile, width=width * 0.9,
                label=f"Cluster {c_id}", color=CLUSTER_COLORS[c_id], alpha=0.8)
axes[1].set_xticks(x_pos + width * (best_k - 1) / 2)
axes[1].set_xticklabels(band_names)
axes[1].set_xlabel("Banda"); axes[1].set_ylabel("Potenza relativa media")
axes[1].set_title("Profilo bande medio per cluster")
axes[1].legend(title="Cluster", fontsize=8)
axes[1].grid(axis="y", alpha=0.4)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_band_profile_per_cluster.png", dpi=120, bbox_inches="tight")
plt.show()

# ANOVA per banda tra cluster
print("\nANOVA tra cluster per banda (potenza media per soggetto):")
print(f"{'Banda':>8}  {'F':>8}  {'p':>10}  {'Sig':>5}")
for b_idx, b_name in enumerate(band_names):
    groups = [band_power_mat[FINAL_CLUSTER_LABELS == c_id, :, b_idx].mean(axis=1)
              for c_id in range(best_k)]
    F, p = f_oneway(*groups)
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
    print(f"{b_name:>8}  {F:>8.3f}  {p:>10.4f}  {sig:>5}")

In [ ]:
# ============================================================
# OVERLAP CON PERFORMANCE EEG_06
# Carica RESULTS_CSV e unisce con cluster label
# ============================================================

results_path = Path(EEG06_RESULTS_CSV)
if not results_path.is_absolute():
    results_path = project_root / results_path

if not results_path.exists():
    # Cerca alternative comuni
    for alt in [
        project_root / "data" / "interim" / "eeg06_results.csv",
        project_root / "data" / "interim" / "results_subject_specific.csv",
    ]:
        if alt.exists():
            results_path = alt
            break

if not results_path.exists():
    print(f"⚠️  File risultati EEG_06 non trovato: {results_path}")
    print("   Esegui prima EEG_06_subject_specific.ipynb per generare i risultati.")
    print("   Questo blocco verrà saltato.")
    HAS_EEG06 = False
else:
    df_results = pd.read_csv(results_path)
    print(f"Caricati risultati EEG_06: {len(df_results)} righe")
    print(f"Colonne: {list(df_results.columns)}")
    print(df_results.head())
    HAS_EEG06 = True

In [ ]:
# ============================================================
# BOXPLOT ACCURACY PER CLUSTER + ANOVA
# ============================================================

if not HAS_EEG06:
    print("Salto: nessun file EEG_06 trovato.")
else:
    # ---- Costruisci mapping soggetto → cluster ----
    subj2cluster = {s: int(FINAL_CLUSTER_LABELS[i]) for i, s in enumerate(subject_ids)}

    # Normalizza ID soggetto nel CSV risultati
    # Il CSV potrebbe avere "P003", "003", "3", ecc.
    def normalize_subj_id(sid):
        """Porta a formato 'PXXX' (3 cifre)"""
        sid_str = str(sid).lstrip("P").zfill(3)
        return f"P{sid_str}"

    df_results["subj_norm"] = df_results["subject"].apply(
        lambda x: normalize_subj_id(x) if "subject" in df_results.columns else None
    )

    # Cerca colonna soggetto
    subj_col = None
    for col in ["subject", "subj", "subject_id", "Subject"]:
        if col in df_results.columns:
            subj_col = col
            break
    acc_col = None
    for col in ["test_acc", "accuracy", "acc", "test_accuracy"]:
        if col in df_results.columns:
            acc_col = col
            break

    if subj_col is None or acc_col is None:
        print(f"❌ Colonne non trovate. Colonne disponibili: {list(df_results.columns)}")
        print(f"   Imposta subj_col e acc_col manualmente.")
        HAS_EEG06 = False
    else:
        df_results["subj_norm"] = df_results[subj_col].apply(normalize_subj_id)
        df_results["cluster"]   = df_results["subj_norm"].map(subj2cluster)

        n_missing = df_results["cluster"].isna().sum()
        if n_missing > 0:
            print(f"⚠️  {n_missing} righe senza cluster (soggetto non in subject_ids) — rimosse.")
        df_merge = df_results.dropna(subset=["cluster"]).copy()
        df_merge["cluster"] = df_merge["cluster"].astype(int)

        print(f"Righe con cluster assegnato: {len(df_merge)}")
        print(f"\nAccuracy media per cluster:")
        print(df_merge.groupby("cluster")[acc_col].agg(["mean", "std", "count"]).round(4))

        # ---- Plot boxplot accuracy per cluster ----
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"Performance EEG_06 per Cluster (k={best_k})", fontsize=13)

        # Boxplot
        cluster_data = [df_merge[df_merge["cluster"] == c][acc_col].values
                        for c in range(best_k)]
        bp = axes[0].boxplot(cluster_data, patch_artist=True,
                              labels=[f"Cluster {c}\n(n={len(d)})" for c, d in enumerate(cluster_data)])
        for patch, color in zip(bp["boxes"], CLUSTER_COLORS):
            patch.set_facecolor(color); patch.set_alpha(0.7)
        # Sovrapponi i punti
        for c_id, data_c in enumerate(cluster_data):
            x_jitter = np.random.default_rng(c_id).uniform(-0.2, 0.2, len(data_c))
            axes[0].scatter((c_id + 1) + x_jitter, data_c,
                             color=CLUSTER_COLORS[c_id], edgecolors="black",
                             linewidths=0.5, alpha=0.7, s=30)
        axes[0].axhline(1 / len(df_merge["cluster"].unique()),  # chance level circa
                         color="gray", ls=":", label="≈Chance")
        axes[0].set_xlabel("Cluster"); axes[0].set_ylabel(f"{acc_col}")
        axes[0].set_title("Accuracy per cluster"); axes[0].legend()
        axes[0].grid(axis="y", alpha=0.4)

        # Scatter: connettività PCC media vs accuracy
        subj_acc_mean = df_merge.groupby("subj_norm")[acc_col].mean().to_dict()
        x_conn, y_acc, c_labels = [], [], []
        for i, s in enumerate(subject_ids):
            if s in subj_acc_mean:
                x_conn.append(mean_connectivity[i])
                y_acc.append(subj_acc_mean[s])
                c_labels.append(FINAL_CLUSTER_LABELS[i])
        x_conn, y_acc, c_labels = map(np.array, [x_conn, y_acc, c_labels])

        for c_id in range(best_k):
            m = c_labels == c_id
            axes[1].scatter(x_conn[m], y_acc[m], s=80, label=f"Cluster {c_id}",
                             color=CLUSTER_COLORS[c_id], edgecolors="black", lw=0.5, alpha=0.85)
        # Linea di tendenza
        if len(x_conn) > 2:
            z = np.polyfit(x_conn, y_acc, 1)
            p_line = np.poly1d(z)
            xs = np.linspace(x_conn.min(), x_conn.max(), 50)
            axes[1].plot(xs, p_line(xs), "k--", lw=1.5, label=f"Trend (slope={z[0]:.3f})")
        axes[1].set_xlabel("Connettività PCC media")
        axes[1].set_ylabel(f"{acc_col} (media soggetto)")
        axes[1].set_title("Connettività vs Accuracy")
        axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

        plt.tight_layout()
        fig.savefig(FIGURES_DIR / "eeg08_accuracy_per_cluster.png", dpi=120, bbox_inches="tight")
        plt.show()

        # ANOVA
        if len(cluster_data) > 1 and all(len(d) > 0 for d in cluster_data):
            F, p_anova = f_oneway(*cluster_data)
            _, p_kw    = kruskal(*cluster_data)
            print(f"\nANOVA accuracy tra cluster: F={F:.3f}, p={p_anova:.4f}")
            print(f"Kruskal-Wallis           : p={p_kw:.4f}")
            if p_anova < 0.05:
                print("✅ Differenza significativa tra cluster (p<0.05)")
            else:
                print("⚪ Differenza NON significativa — cluster simili in performance")

In [ ]:
# ============================================================
# HEATMAP PCC MEDIA PER CLUSTER
# Caratterizzazione neurofisiologica di ogni cluster
# ============================================================

fig, axes = plt.subplots(1, best_k, figsize=(6 * best_k, 5))
if best_k == 1:
    axes = [axes]
fig.suptitle(f"PCC Medio per Cluster (k={best_k})", fontsize=12)

pcc_vmax = float(np.percentile(pcc_mean_mat[np.triu_indices(n_subj, k=1)[0],
                                             np.triu_indices(n_subj, k=1)[1]], 95)) \
           if False else 0.5

for c_id in range(best_k):
    mask = FINAL_CLUSTER_LABELS == c_id
    pcc_c = pcc_mean_mat[mask].mean(axis=0)   # (N_CHANS, N_CHANS)
    im = axes[c_id].imshow(pcc_c, cmap="RdYlBu_r", vmin=0, vmax=0.5)
    axes[c_id].set_title(f"Cluster {c_id}\n(n={mask.sum()} soggetti)")
    axes[c_id].set_xlabel("Canale"); axes[c_id].set_ylabel("Canale")
    plt.colorbar(im, ax=axes[c_id], label="|PCC|")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eeg08_pcc_per_cluster.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# SOMMARIO E EXPORT
# ============================================================

print("=" * 65)
print(f"SOMMARIO CLUSTERING SOGGETTI (EEG_08)")
print("=" * 65)
print(f"Soggetti analizzati    : {n_subj}")
print(f"Feature per soggetto   : {X_subjects.shape[1]} "
      f"({N_CHANS}×{n_bands} band power + {n_triu} PCC + {n_graph} graph)")
print(f"Migliore k (silhouette): {best_k} (score={silhouette_scores[best_k]:.4f})")
print("")
print("Cluster:")
for c_id in range(best_k):
    mask   = FINAL_CLUSTER_LABELS == c_id
    n_c    = mask.sum()
    conn_c = mean_connectivity[mask].mean()
    gam_c  = bp_gamma_per_subj[mask].mean() if 'bp_gamma_per_subj' in dir() else float('nan')
    members = [subject_ids[i] for i in range(n_subj) if FINAL_CLUSTER_LABELS[i] == c_id]
    print(f"  Cluster {c_id}: n={n_c:2d}  conn_pcc={conn_c:.4f}  gamma={gam_c:.4f}")
    print(f"    Soggetti: {', '.join(members)}")

print("")
print("Artefatti salvati:")
for f in sorted(FIGURES_DIR.glob("eeg08_*.png")):
    print(f"  {f.name}")

# ---- Export CSV: soggetto → cluster → feature principali ----
df_export = pd.DataFrame({
    "subject":          subject_ids,
    "cluster":          FINAL_CLUSTER_LABELS,
    "mean_pcc_conn":    mean_connectivity,
    "std_pcc_conn":     std_connectivity,
    "n_trials":         [n_trials_per.get(s, 0) for s in subject_ids],
})
# Aggiungi potenza per banda
for b_idx, b_name in enumerate(band_names):
    df_export[f"bp_{b_name}"] = band_power_mat[:, :, b_idx].mean(axis=1)

export_path = INTERIM_DIR / "subject_clusters.csv"
df_export.to_csv(export_path, index=False)
print(f"\nTabella cluster esportata: {export_path}")
print(df_export.to_string(index=False))